# mini-qforest — 02. Graph neural networks

**Question:** if each image is connected to the images that look most like it, does a graph neural network (GNN) classify better than a model that sees each image alone?

**Method:**
1. each of the 27,000 images is a **node**, described by its 35 features from notebook 01 (or its CNN embedding);
2. each node is connected to its *k* nearest neighbours in feature space (k-NN graph);
3. we compare an MLP (no graph), a **GCN** and a **GAT** on the same train/val/test split as notebook 01.

This is semi-supervised node classification in the *transductive* setting: all nodes and edges are visible during training, only the training labels are used.

> Run notebook 01 first: it creates `features.npz` and `split.npz` in your Drive.

In [ ]:
!pip install -q torch_geometric

In [ ]:
# --- Setup: works in Google Colab (results saved to your Google Drive) or locally ---
import os, sys, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/mini-qforest"
else:
    WORKDIR = os.path.abspath("..")
DATA_DIR = os.path.join(WORKDIR, "data")
RES_DIR = os.path.join(WORKDIR, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
SEED = 42
print("Working directory:", WORKDIR)

In [ ]:
# --- Helper: every model writes its score to results/results.csv (shared by the 3 notebooks) ---
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

RESULTS_CSV = os.path.join(RES_DIR, "results.csv")

def log_result(model, task, y_true, y_pred, n_params=None, train_time_s=None):
    binary = len(np.unique(y_true)) == 2
    row = {
        "model": model,
        "task": task,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred, average="binary" if binary else "macro"), 4),
        "n_params": n_params,
        "train_time_s": None if train_time_s is None else round(train_time_s, 1),
    }
    df = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=list(row))
    df = df[~((df["model"] == model) & (df["task"] == task))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RESULTS_CSV, index=False)
    print(row)
    return row

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

d = np.load(os.path.join(RES_DIR, "features.npz"), allow_pickle=True)
y10, ybin, classes = d["y10"], d["ybin"], list(d["classes"])
s = np.load(os.path.join(RES_DIR, "split.npz"))
idx_train, idx_val, idx_test = s["train"], s["val"], s["test"]

NODE_FEATURES = "handcrafted"   # "handcrafted" (35 features) or "cnn" (128-d CNN embeddings)
X_raw = d["X"] if NODE_FEATURES == "handcrafted" else np.load(os.path.join(RES_DIR, "cnn_embeddings.npy"))
X = StandardScaler().fit(X_raw[idx_train]).transform(X_raw).astype(np.float32)
N = len(X)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(N, "nodes,", X.shape[1], "features, device:", device)

## 1. Build the k-NN graph

Edges are made symmetric (if A is a neighbour of B, B is a neighbour of A).
**Homophily** = share of edges that link two images of the same class. The higher it is, the more a GNN can gain by aggregating neighbours.

In [ ]:
from sklearn.neighbors import kneighbors_graph

def knn_edge_index(X, k):
    A = kneighbors_graph(X, n_neighbors=k, mode="connectivity", include_self=False, n_jobs=-1)
    A = A.maximum(A.T).tocoo()
    return torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)

def homophily(edge_index, y):
    src, dst = edge_index.numpy()
    return float((y[src] == y[dst]).mean())

K = 10
edge_index = knn_edge_index(X, K)
print(f"k={K}: {edge_index.shape[1]:,} directed edges, mean degree {edge_index.shape[1] / N:.1f}")
print(f"homophily 10 classes: {homophily(edge_index, y10):.3f} | forest vs rest: {homophily(edge_index, ybin):.3f}")

In [ ]:
from torch_geometric.data import Data

def mask(ids):
    m = torch.zeros(N, dtype=torch.bool); m[torch.from_numpy(ids)] = True; return m

def make_data(edge_index):
    data = Data(x=torch.from_numpy(X), edge_index=edge_index)
    data.train_mask, data.val_mask, data.test_mask = mask(idx_train), mask(idx_val), mask(idx_test)
    return data

data = make_data(edge_index)
data

## 2. Models

Same size for all three (hidden dimension 64, two layers, dropout 0.5), so that differences come from the graph, not from capacity.

In [ ]:
from torch_geometric.nn import GCNConv, GATConv

class MLP(nn.Module):
    def __init__(self, d_in, d_out, h=64):
        super().__init__(); self.l1, self.l2 = nn.Linear(d_in, h), nn.Linear(h, d_out)
    def forward(self, x, edge_index=None):
        x = Fn.dropout(Fn.relu(self.l1(x)), 0.5, self.training)
        return self.l2(x)

class GCN(nn.Module):
    def __init__(self, d_in, d_out, h=64):
        super().__init__(); self.c1, self.c2 = GCNConv(d_in, h), GCNConv(h, d_out)
    def forward(self, x, edge_index):
        x = Fn.dropout(Fn.relu(self.c1(x, edge_index)), 0.5, self.training)
        return self.c2(x, edge_index)

class GAT(nn.Module):
    def __init__(self, d_in, d_out, h=16, heads=4):
        super().__init__()
        self.c1 = GATConv(d_in, h, heads=heads, dropout=0.2)
        self.c2 = GATConv(h * heads, d_out, heads=1, dropout=0.2)
    def forward(self, x, edge_index):
        x = Fn.dropout(Fn.elu(self.c1(x, edge_index)), 0.5, self.training)
        return self.c2(x, edge_index)

def n_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
def train_eval(model, data, y, epochs=300, lr=0.01, weight_decay=5e-4):
    """Full-batch training; keeps the epoch with the best validation accuracy."""
    model, data = model.to(device), data.to(device)
    y = torch.from_numpy(y).long().to(device)
    counts = torch.bincount(y[data.train_mask]).float()
    class_w = (counts.sum() / (len(counts) * counts)).to(device)   # balances forest vs rest
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_state, curve = -1.0, None, []
    t0 = time.time()
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = Fn.cross_entropy(out[data.train_mask], y[data.train_mask], weight=class_w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(data.x, data.edge_index).argmax(1)
        val_acc = (pred[data.val_mask] == y[data.val_mask]).float().mean().item()
        curve.append((loss.item(), val_acc))
        if val_acc > best_val:
            best_val, best_state = val_acc, {k: v.detach().clone() for k, v in model.state_dict().items()}
    train_time = time.time() - t0
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(1).cpu().numpy()
    return pred[idx_test], curve, train_time

## 3. Results on both tasks

In [ ]:
curves = {}
for task, y in [("forest_vs_rest", ybin), ("10_classes", y10)]:
    n_out = int(y.max()) + 1
    for name, Model in [("MLP (no graph)", MLP), (f"GCN (k-NN graph, k={K})", GCN), (f"GAT (k-NN graph, k={K})", GAT)]:
        torch.manual_seed(SEED)
        model = Model(X.shape[1], n_out)
        pred, curve, dt = train_eval(model, data, y)
        log_result(f"{name} [{NODE_FEATURES}]", task, y[idx_test], pred, n_params(model), dt)
        curves[(task, name)] = curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, task in zip(axes, ["forest_vs_rest", "10_classes"]):
    for (t, name), curve in curves.items():
        if t == task:
            ax.plot([c[1] for c in curve], label=name)
    ax.set_title(task); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].set_ylabel("validation accuracy"); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "gnn_curves.png"), dpi=150); plt.show()

## 4. Sensitivity to k

How many neighbours should each node have? Too few: little information to aggregate. Too many: edges to dissimilar images (lower homophily).

In [ ]:
rows = []
for k in [3, 5, 10, 20, 40]:
    ei = knn_edge_index(X, k)
    torch.manual_seed(SEED)
    pred, _, _ = train_eval(GCN(X.shape[1], 10), make_data(ei), y10)
    rows.append({"k": k, "homophily": round(homophily(ei, y10), 3),
                 "GCN test accuracy (10 classes)": round((pred == y10[idx_test]).mean(), 4)})
sens = pd.DataFrame(rows); sens.to_csv(os.path.join(RES_DIR, "gnn_k_sensitivity.csv"), index=False)
sens

In [ ]:
pd.read_csv(RESULTS_CSV)

## Discussion (to complete with your results)

- Does the graph help compared with the MLP? On which task?
- How does accuracy follow homophily when *k* changes?
- Try `NODE_FEATURES = "cnn"` and rerun: are CNN embeddings better node features?

**Link with the PhD (Q-FOREST):** here the graph is built from image similarity, because EuroSAT patches have no coordinates.
In the thesis, graphs can be **spatial**: forest parcels or image segments as nodes, adjacency as edges, multimodal data (optical, radar, LiDAR) as node features. Old-growth forest is defined by a patch *and its context*, which is exactly what message passing captures.

Next: **03_hybrid_quantum.ipynb** tests whether a small quantum circuit can compete with classical models of the same size.